In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer('all-MiniLM-L6-v2')

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
from src import RAGBase, OllamaClient
from dotenv import load_dotenv

load_dotenv()
ollama_client = OllamaClient()

class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self._course_filter}

        return self._index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm=ollama_client,
    model="granite4.1:3b",
    course_filter="data-engineering-zoomcamp"
)

In [12]:
vector_assistant._course_filter

'data-engineering-zoomcamp'

In [13]:
vector_assistant.rag('the program has already begun, can I still sign up?')

'Yes, you can still join the course even after it has started. The context indicates that participation is open to anyone who wishes to submit homework, as long as there are deadlines for assignments and final projects. Additionally, all materials will remain available for review after the course concludes, allowing you to continue learning at your own pace.'

In [14]:
vs_index.close()